In [1]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, TextIteratorStreamer
import torch
import gc
from dotenv import load_dotenv
import os
import gradio as gr
import torch, threading

In [2]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)

hf_token = os.getenv('HUGGING_FACE_CLAIRE')
if hf_token:
    print(f"Hugging Face Key exists and begins {hf_token[:8]}")
else:
    print(f"Hugging Face Key does not exists.")

#login into hugging face
login(token=hf_token, add_to_git_credential=True)

Hugging Face Key exists and begins hf_tnTKF


In [3]:
# LLAMA= "meta-llama/Llama-3.2-1B-Instruct"
LLAMA= "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [4]:
system_prompt = f"""
  You will act as a specialist in data generation to write tests to evaluate RAG. The file will be provided by the user as the number of tests you should create. Consider information only from these files. Do not create information or use other source than the file provided. 
"""

In [ ]:
# TODO: let the user choose the model (example) for questions
def data_user_prompt(number_of_data, format_type, file_content):

    example = {
        "question": "Who won the prestigious IIOTY award in 2023?",
        "keywords": ["Maxine", "Thompson", "IIOTY"],
        "reference_answer": "Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.",
        "category": "direct_fact"
    }

    user_prompt = f"""
        You are a dataset creator.

        Use the following document as the ONLY source of truth to generate evaluation data for a Retrieval-Augmented Generation (RAG) system.

        <Document>
        {file_content}
        </Document>

        Create {number_of_data} examples in {format_type} format.

        Follow this example structure:
        {example}

        Rules:
        - Do NOT repeat the example
        - Do NOT create questions outside the document
        - Each question must be answerable from the document
        - Return ONLY the dataset.
        - Do not describe the process.
        - Do not mention scripts or file saving.
        """

    return user_prompt

In [6]:
import requests
print(requests.get("https://huggingface.co").status_code)

200


In [7]:
# Quantization Config - this allows us to load the model into memory and use less memory - reduce the precision
quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [8]:
# Tokenizer messages
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token #End Of Sentence Token
# inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

In [9]:
# The model - create model, it connects to hugging face and download all the model weights and put in cache/memory, when disconnect it will be deleted
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [10]:
def generate(model, messages, quant=True, max_new_tokens=80):  
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
  attention_mask = torch.ones_like(input_ids, device="cuda")
  outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens)
  output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
  
  return output_text

In [ ]:
# TODO: Add save to file .json or .csv 
def process_file(number_of_data, format_type, file):
    # Read uploaded file contents
    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        file_content = f.read()

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": data_user_prompt(number_of_data, format_type, file_content)},
    ]

    # STREAMING version
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to("cuda")

    streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True)
    generation_kwargs = dict(inputs=inputs, max_new_tokens=2000, streamer=streamer)

    thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    output_text = "".join(token for token in streamer)

    return output_text

In [12]:
def gradio_interface(number_of_data, format_type, file):
    return process_file(number_of_data, format_type, file)

In [13]:
default_number_of_data = 1000

In [ ]:
with gr.Blocks() as demo:

    file_input = gr.File(label="Upload your file", type="filepath")

    num_input = gr.Number(
        label="Number of Examples",
        value=default_number_of_data,
        precision=0
    )

    format_input = gr.Radio(
        choices=["json", "csv"],
        value="json",
        label="Output format"
    )

    generate_btn = gr.Button("Generate")

    output = gr.Textbox(label="Generated Dataset", lines=15)

    copy_btn = gr.Button("📋 Copy")

    # Run main function
    generate_btn.click(
        gradio_interface,
        inputs=[num_input, format_input, file_input],
        outputs=output
    )

    # Copy to clipboard
    copy_btn.click(
        None,
        inputs=output,
        outputs=None,
        js="(text) => navigator.clipboard.writeText(text)"
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "d:\AI\.venv\Lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\AI\.venv\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\AI\.venv\Lib\site-packages\fastapi\applications.py", line 1054, in __call__
    await super().__call__(scope, receive, send)
  File "d:\AI\.venv\Lib\site-packages\starlette\applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "d:\AI\.venv\Lib\site-packages\starlette\middleware\errors.py", line 186, in __call__
    raise exc
  File "d:\AI\.venv\Lib\site-packages\starlette\middleware\errors.py", line 164, in __call__
    await self.app(scope, recei